## (*) **WRITE YOUR NAME**, course number and assignment number here
<br>  
<br>


# CSC 583 HW#2: Exploring Word Vectors (Total 5 points)

This code is adopted almost verbatim from Assign #1 from Stanford University, CS224n Natural Language Processing with Deep Learning course (https://web.stanford.edu/class/cs224n/).

===========================================================

Before you start, make sure you **read the README.md** in the same directory as this notebook for important setup information. You need to install some Python libraries before you can successfully do this assignment. A lot of code is provided in this notebook, and we highly encourage you to read and understand it as part of the learning :)

If you aren't super familiar with Python, Numpy, or Matplotlib, the CS231N Python/Numpy [tutorial](https://cs231n.github.io/python-numpy-tutorial/) is a great resource.

**Assignment Notes:** Be sure to read the <font color='blue'>submission instructions</font> located at the bottom of the notebook.


In [ ]:
!pip install gensim
!pip install datasets

In [ ]:
# All Import Statements Defined Here
# Note: Do not add to this list.
# ----------------

import sys
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 8

from platform import python_version
assert int(python_version().split(".")[1]) >= 5, "Please upgrade your Python version following the instructions in \
    the README.md file found in the same directory as this notebook. Your Python version is " + python_version()

from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

from datasets import load_dataset
imdb_dataset = load_dataset("stanfordnlp/imdb", name="plain_text")

import re
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA

START_TOKEN = '<START>'
END_TOKEN = '<END>'
NUM_SAMPLES = 150

np.random.seed(0)
random.seed(0)
# ----------------

## Part 1: Count-Based Word Vectors (2 points)

Recall the distributional hypothesis: words used in similar contexts tend to have similar meaning. A **co-occurrence matrix** $M$ counts, for a fixed window size $n$, how often word $w_j$ occurs within $n$ words of $w_i$ across the corpus ($M_{ij}$).

**Example (window size $n=1$):**

Document 1: "`<START>` the cat sat on the mat `<END>`"
Document 2: "`<START>` the dog sat on the mat `<END>`"

|          | `<END>` | `<START>` | cat | dog | mat | on  | sat | the |
|----------|---------|-------|-----|-----|-----|-----|-----|-----|
| `<END>`      | 0       | 0     | 0   | 0   | 2   | 0   | 0   | 0   |
| `<START>`    | 0       | 0     | 0   | 0   | 0   | 0   | 0   | 2   |
| cat      | 0       | 0     | 0   | 0   | 0   | 0   | 1   | 1   |
| dog      | 0       | 0     | 0   | 0   | 0   | 0   | 1   | 1   |
| mat      | 2       | 0     | 0   | 0   | 0   | 0   | 0   | 2   |
| on       | 0       | 0     | 0   | 0   | 0   | 0   | 2   | 2   |
| sat      | 0       | 0     | 1   | 1   | 0   | 2   | 0   | 0   |
| the      | 0       | 2     | 1   | 1   | 2   | 2   | 0   | 0   |

Notice that **cat** and **dog** get *identical* rows: both only ever appear as "the `___` sat", so they share the exact same context words. This is the distributional hypothesis in miniature â€” words used the same way end up with the same (or, in a real corpus, merely similar) vector, which is exactly why we can use these vectors as a proxy for meaning.

Rows of $M$ are word vectors, but high-dimensional and sparse. We reduce dimensionality with **truncated SVD** (top $k$ singular components), which preserves this closeness (e.g. *cat* stays closer to *dog* than to *sat*) in far fewer dimensions. Use the `sklearn`/`scipy` implementation rather than writing your own SVD.

### Corpus

We use 150 documents from the IMDB movie review dataset. `read_corpus` lowercases words, strips non-word characters, and wraps each document with `<START>`/`<END>` tokens.

In [ ]:
def read_corpus():
    """ Read files from the Large Movie Review Dataset.
        Params:
            category (string): category name
        Return:
            list of lists, with words from each of the processed files
    """
    files = imdb_dataset["train"]["text"][:NUM_SAMPLES]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]

In [ ]:
imdb_corpus = read_corpus()
pprint.pprint(imdb_corpus[:3], compact=True, width=100)
print("corpus size: ", len(imdb_corpus[0]))

### Question 1.1: Implement `distinct_words` [code] (0.4 points)

Return the sorted list of distinct words (types) in `corpus`, and their count. (List comprehensions + `set` will make this fast; see [flattening a list of lists](https://coderwall.com/p/rcmaea/flatten-a-list-of-lists-in-one-line-in-python).)

In [ ]:
def distinct_words(corpus):
    """ Determine a list of distinct words for the corpus.
        Params:
            corpus (list of list of strings): corpus of documents
        Return:
            corpus_words (list of strings): sorted list of distinct words across the corpus
            n_corpus_words (integer): number of distinct words across the corpus
    """
    corpus_words = []
    n_corpus_words = -1

    # ------------------
    # Write your implementation here.

    # ------------------

    return corpus_words, n_corpus_words

In [ ]:
# ---------------------
# Run this sanity check
# Note that this not an exhaustive check for correctness.
# ---------------------

# Define toy corpus
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
test_corpus_words, num_corpus_words = distinct_words(test_corpus)

# Correct answers
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
ans_num_corpus_words = len(ans_test_corpus_words)

# Test correct number of words
assert(num_corpus_words == ans_num_corpus_words), "Incorrect number of distinct words. Correct: {}. Yours: {}".format(ans_num_corpus_words, num_corpus_words)

# Test correct words
assert (test_corpus_words == ans_test_corpus_words), "Incorrect corpus_words.\nCorrect: {}\nYours:   {}".format(str(ans_test_corpus_words), str(test_corpus_words))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### Question 1.2: Implement `compute_co_occurrence_matrix` [code] (0.6 points)

Write a method that constructs a co-occurrence matrix for a certain window-size $n$ (with a default of 4), considering words $n$ before and $n$ after the word in the center of the window. Here, we start to use `numpy (np)` to represent vectors, matrices, and tensors.


In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """ Compute co-occurrence matrix for the given corpus and window_size (default of 4).

        Note: Each word in a document should be at the center of a window. Words near edges will have a smaller
              number of co-occurring words.

              For example, if we take the document "<START> All that glitters is not gold <END>" with window size of 4,
              "All" will co-occur with "<START>", "that", "glitters", "is", and "not".

        Params:
            corpus (list of list of strings): corpus of documents
            window_size (int): size of context window
        Return:
            M (a symmetric numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)):
                Co-occurence matrix of word counts.
                The ordering of the words in the rows/columns should be the same as the ordering of the words given by the distinct_words function.
            word2ind (dict): dictionary that maps word to index (i.e. row/column number) for matrix M.
    """
    words, n_words = distinct_words(corpus)
    M = None
    word2ind = {}

    # ------------------
    # Write your implementation here.

    # ------------------

    return M, word2ind

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# ---------------------

# Define toy corpus and get student's co-occurrence matrix
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)

# Correct M and word2ind
M_test_ans = np.array(
    [[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,],
     [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,],
     [1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,],
     [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.,],
     [0., 0., 1., 0., 1., 1., 0., 0., 0., 1.,],
     [1., 0., 0., 1., 1., 0., 0., 0., 1., 0.,]]
)
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
word2ind_ans = dict(zip(ans_test_corpus_words, range(len(ans_test_corpus_words))))

# Test correct word2ind
assert (word2ind_ans == word2ind_test), "Your word2ind is incorrect:\nCorrect: {}\nYours: {}".format(word2ind_ans, word2ind_test)

# Test correct M shape
assert (M_test.shape == M_test_ans.shape), "M matrix has incorrect shape.\nCorrect: {}\nYours: {}".format(M_test.shape, M_test_ans.shape)

# Test correct M values
for w1 in word2ind_ans.keys():
    idx1 = word2ind_ans[w1]
    for w2 in word2ind_ans.keys():
        idx2 = word2ind_ans[w2]
        student = M_test[idx1, idx2]
        correct = M_test_ans[idx1, idx2]
        if student != correct:
            print("Correct M:")
            print(M_test_ans)
            print("Your M: ")
            print(M_test)
            raise AssertionError("Incorrect count at index ({}, {})=({}, {}) in matrix M. Yours has {} but should have {}.".format(idx1, idx2, w1, w2, student, correct))

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### Question 1.3: Implement `reduce_to_k_dim` [code] (0.4 point)

Use [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) (it's the only common implementation with an efficient randomized algorithm for large-scale truncated SVD) to reduce $M$ to $k$ dimensions, returning $U \cdot S$.

In [ ]:
def reduce_to_k_dim(M, k=2):
    """ Reduce a co-occurence count matrix of dimensionality (num_corpus_words, num_corpus_words)
        to a matrix of dimensionality (num_corpus_words, k) using the following SVD function from Scikit-Learn:
            - http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html

        Params:
            M (numpy matrix of shape (number of unique words in the corpus , number of unique words in the corpus)): co-occurence matrix of word counts
            k (int): embedding size of each word after dimension reduction
        Return:
            M_reduced (numpy matrix of shape (number of corpus words, k)): matrix of k-dimensioal word embeddings.
                    In terms of the SVD from math class, this actually returns U * S
    """
    n_iters = 10    # Use this parameter in your call to `TruncatedSVD`
    M_reduced = None
    print("Running Truncated SVD over %i words..." % (M.shape[0]))

    # ------------------
    # Write your implementation here.


    # ------------------

    print("Done.")
    return M_reduced

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness
# In fact we only check that your M_reduced has the right dimensions.
# ---------------------

# Define toy corpus and run student code
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)
M_test_reduced = reduce_to_k_dim(M_test, k=2)

# Test proper dimensions
assert (M_test_reduced.shape[0] == 10), "M_reduced has {} rows; should have {}".format(M_test_reduced.shape[0], 10)
assert (M_test_reduced.shape[1] == 2), "M_reduced has {} columns; should have {}".format(M_test_reduced.shape[1], 2)

# Print Success
print ("-" * 80)
print("Passed All Tests!")
print ("-" * 80)

### Question 1.4: Implement `plot_embeddings` [code] (0.4 point)

Scatter-plot the rows of `M_reduced` corresponding to `words` (using `matplotlib`), labeling each point with its word.

In [ ]:
def plot_embeddings(M_reduced, word2ind, words):
    """ Plot in a scatterplot the embeddings of the words specified in the list "words".
        NOTE: do not plot all the words listed in M_reduced / word2ind.
        Include a label next to each point.

        Params:
            M_reduced (numpy matrix of shape (number of unique words in the corpus , 2)): matrix of 2-dimensioal word embeddings
            word2ind (dict): dictionary that maps word to indices for matrix M
            words (list of strings): words whose embeddings we want to visualize
    """

    # ------------------
    # Write your implementation here.

    # ------------------

In [ ]:
# ---------------------
# Run this sanity check
# Note that this is not an exhaustive check for correctness.
# The plot produced should look like the included file question_1.4_test.png
# ---------------------

print ("-" * 80)
print ("Outputted Plot:")

M_reduced_plot_test = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1], [0, 0]])
word2ind_plot_test = {'test1': 0, 'test2': 1, 'test3': 2, 'test4': 3, 'test5': 4}
words = ['test1', 'test2', 'test3', 'test4', 'test5']
plot_embeddings(M_reduced_plot_test, word2ind_plot_test, words)

print ("-" * 80)

### Question 1.5: Co-Occurrence Plot Analysis [written] (0.5 points)

Run the cell below to compute the co-occurrence matrix (window size 4) over the IMDB corpus, reduce it to 2D with `reduce_to_k_dim`, normalize each vector to unit length (so closeness is directional), and plot it. This can take a few minutes.

**Verify your figure matches `question_1.5.png` in the assignment zip; use that figure if yours doesn't match.**

In [ ]:
# -----------------------------
# Run This Cell to Produce Your Plot
# ------------------------------
imdb_corpus = read_corpus()
M_co_occurrence, word2ind_co_occurrence = compute_co_occurrence_matrix(imdb_corpus)
M_reduced_co_occurrence = reduce_to_k_dim(M_co_occurrence, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced_co_occurrence, axis=1)
M_normalized = M_reduced_co_occurrence / M_lengths[:, np.newaxis] # broadcasting

words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_normalized, word2ind_co_occurrence, words)

a. Find at least two groups of words that cluster together in 2-dimensional embedding space. Give an explanation for each cluster you observe.

#### <font color="red">Write your answer here.</font>


b. What doesn't cluster together that you might think should have? Describe at least two examples.

#### <font color="red">Write your answer here.</font>

## Part 2: Prediction-Based Word Vectors (3 points)

Prediction-based methods (word2vec, GloVe) generally outperform count-based ones. Here we explore **GloVe** embeddings. Run the cell below to load them (first run downloads the model, ~a couple minutes; cached afterward). If you get a "reset by peer" error, just rerun the cell.

In [ ]:
def load_embedding_model():
    """ Load GloVe Vectors
        Return:
            wv_from_bin: All 400000 embeddings, each length 200
    """
    import gensim.downloader as api
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print("Loaded vocab size %i" % len(list(wv_from_bin.index_to_key)))
    return wv_from_bin

# Load the GloVe embeddings
wv_from_bin = load_embedding_model()

### Reducing Dimensionality of Word Embeddings

To directly compare against Part 1, and to avoid running out of memory, we sample 40000 GloVe vectors, put them in a matrix, and reuse your `reduce_to_k_dim` to project the 200-dim vectors to 2D.

In [ ]:
def get_matrix_of_vectors(wv_from_bin, required_words):
    """ Put the GloVe vectors into a matrix M.
        Param:
            wv_from_bin: KeyedVectors object; the 400000 GloVe vectors loaded from file
        Return:
            M: numpy matrix shape (num words, 200) containing the vectors
            word2ind: dictionary mapping each word to its row number in M
    """
    import random
    words = list(wv_from_bin.index_to_key)
    print("Shuffling words ...")
    random.seed(225)
    random.shuffle(words)
    print("Putting %i words into word2ind and matrix M..." % len(words))
    word2ind = {}
    M = []
    curInd = 0
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    for w in required_words:
        if w in words:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    M = np.stack(M)
    print("Done.")
    return M, word2ind

In [ ]:
# -----------------------------------------------------------------
# Run Cell to Reduce 200-Dimensional Word Embeddings to k Dimensions
# Note: This should be quick to run
# -----------------------------------------------------------------
M, word2ind = get_matrix_of_vectors(wv_from_bin, words)
M_reduced = reduce_to_k_dim(M, k=2)

# Rescale (normalize) the rows to make them each of unit-length
M_lengths = np.linalg.norm(M_reduced, axis=1)
M_reduced_normalized = M_reduced / M_lengths[:, np.newaxis] # broadcasting

### Question 2.1: GloVe Plot Analysis [written] (0.4 points)

Run the cell below to plot the 2D GloVe embeddings for the same words as Q1.5. **Verify your figure matches `question_2.1.png`.**

In [ ]:
words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_reduced_normalized, word2ind, words)

a. What is one way the plot is different from the one generated earlier from the co-occurrence matrix? What is one way it's similar?

#### <font color="red">Write your answer here.</font>

b. Why might the GloVe plot (question_2.1.png) differ from the plot generated earlier from the co-occurrence matrix (question_1.5.png)?

#### <font color="red">Write your answer here.</font>

### Cosine Similarity

Cosine similarity between vectors $p$ and $q$: $s = \dfrac{p \cdot q}{||p||\,||q||} \in [-1, 1]$. We use this to find words that are "close" and "far" from one another.

### Question 2.2: Words with Multiple Meanings (0.4 points) [code + written]
Polysemes and homonyms are words that have more than one meaning (see this [wiki page](https://en.wikipedia.org/wiki/Polysemy) to learn more about the difference between polysemes and homonyms ). Find a word with *at least two VERY different meanings* such that the top-10 most similar words (according to cosine similarity) contain related words from *both* meanings. For example, a noun <b>"bat"</b> has both "a flying mammal" and "a sports equipment" meanings in the top 10, and a verb <b>"strike"</b> has both "to hit" and "to protest" meanings. You will probably need to try several polysemous or homonymic words before you find one.

<b> Give <u>one noun</u> and <u>one verb</u> (not "bat" or "strike") that are polysemous and have VERY different meanings in the top 10 most similar words</b>.  Your score will depend on your choice of the words and the <b>explanations</b> you write in the comment section below.

**Note**: You should use the `wv_from_bin.most_similar(word)` function to get the top 10 most similar words of a given word. This function ranks all other words in the vocabulary with respect to their cosine similarity to the given word.

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">Write your answer here.</font>

### Question 2.3: Synonyms & Antonyms (0.6 points) [code + written]

Cosine distance = 1 - cosine similarity. Find three words $(w_1, w_2, w_3)$ where $w_1,w_2$ are synonyms and $w_1,w_3$ are antonyms, but cosine distance$(w_1,w_3)$ < cosine distance$(w_1,w_2)$ â€” e.g. "happy" is closer to "sad" than to "cheerful". Find a *different* example (use `wv_from_bin.distance(w1, w2)`, [docs](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.distance)) and give a possible explanation for the result.

In [ ]:
# ------------------
# Write your implementation here.

# ------------------

#### <font color="red">Write your answer here.</font>

### Question 2.4: Analogies with Word Vectors [written] (0.3 points)

Word vectors can sometimes solve analogies. Run the cell below for "man : grandfather :: woman : x" using `most_similar` ([docs](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar)), which ranks words by similarity to the `positive` list and dissimilarity to the `negative` list.

Let $m, g, w, x$ denote the vectors for `man`, `grandfather`, `woman`, and the answer. Using **only** $m, g, w$ and $+/-$, what expression is being maximized in cosine similarity with $x$?

In [ ]:
# Run this cell to answer the analogy -- man : grandfather :: woman : x
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'grandfather'], negative=['man']))

#### <font color="red">Write your answer here.</font>

### Question 2.5: Finding Analogies [code + written]  (0.2 points)
a. For the previous example, it's clear that "grandmother" completes the analogy. But give an intuitive explanation as to why the `most_similar` function gives us words like "granddaughter", "daughter", or "mother?

#### <font color="red">Write your answer here.</font>

b. Find your own analogy that holds (intended word ranked top). State it as x:y :: a:b, and explain briefly if it's non-obvious. You may need to try several.

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------

# Test the solution
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] == b

#### <font color="red">Write your answer here.</font>

### Question 2.6: Incorrect Analogy [code + written] (0.2 points)
a. Below, we expect to see the intended analogy "hand : glove :: foot : **sock**", but we see an unexpected result instead. Give a potential reason as to why this particular analogy turned out the way it did?

In [ ]:
pprint.pprint(wv_from_bin.most_similar(positive=['foot', 'glove'], negative=['hand']))

#### <font color="red">Write your answer here.</font>

b. Find another analogy that does *not* hold. State the intended analogy as x:y :: a:b and the incorrect top result.

In [ ]:
# For example: x, y, a, b = ("", "", "", "")
# ------------------
# Write your implementation here.


# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] != b

#### <font color="red">Write your answer here.</font>

### Question 2.7: Guided Analysis of Bias in Word Vectors [written] (0.2 point)

Word embeddings can encode social biases, which can reinforce stereotypes downstream. Run the cell below to compare words similar to "man"+"profession" (dissimilar to "woman") against words similar to "woman"+"profession" (dissimilar to "man"). Describe the difference between the two lists and how it reflects gender bias.

In [ ]:
# Run this cell
# Here `positive` indicates the list of words to be similar to and `negative` indicates the list of words to be
# most dissimilar from.

pprint.pprint(wv_from_bin.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'profession'], negative=['man']))

#### <font color="red">Write your answer here.</font>

### Question 2.8: Independent Analysis of Bias in Word Vectors [code + written] (0.2 point)

Use `most_similar` to find another example of bias in these vectors, and briefly explain it.

In [ ]:
# ------------------
# Write your implementation here.


# ------------------

#### <font color="red">Write your answer here.</font>

### Question 2.9: Thinking About Bias [written] (0.2 points)

a. Give one possible explanation for how bias enters word vectors specifically (not general AI systems like ChatGPT). Historical examples are welcome but not required.

#### <font color="red">Write your answer here.</font>

b. Suggest one method to mitigate bias in word vectors, and briefly explain the method and its goal.


#### <font color="red">Write your answer here.</font>

## <font color="blue"> Submission Instructions</font>

1. Click the Save button at the top of the Jupyter Notebook.
2. Select Edit -> Clear Outputs of All Cells. This will clear all the outputs from all cells (but will keep the content of all cells).
2. Select Run -> Run All Cells. This will run all the cells in order, and will take several minutes.
3. Submit your Notebook file on D2L.